In [2]:
import sys
sys.path.append('..')
import pandas as pd

# Import your custom biological functions
from src.reverse_complement import reverse_complement
from src.dna_has_stop import has_stop_codon

In [3]:
# 1. Define raw data
raw_data = {
    "clone_id": ["VH_001", "VH_002", "VH_003", "VH_004"],
    "sequence": ["ATGCGTAN", "ATGTGATAG", "CCCGGG", "ATGTGA"] 
}

# 2. Convert to a structured DataFrame
df = pd.DataFrame(raw_data)

# Display the DataFrame
df

,clone_id,sequence
0,VH_001,ATGCGTAN
1,VH_002,ATGTGATAG
2,VH_003,CCCGGG
3,VH_004,ATGTGA


In [4]:
# Calculate length using Python's built-in len function
df['length'] = df['sequence'].apply(len)

# Generate the reverse complement using your module
df['rev_comp'] = df['sequence'].apply(reverse_complement)

# Check for in-frame stop codons using your module
df['has_stop_codon'] = df['sequence'].apply(has_stop_codon)

# View the final, engineered dataset
df

,clone_id,sequence,length,rev_comp,has_stop_codon
0,VH_001,ATGCGTAN,8,NTACGCAT,False
1,VH_002,ATGTGATAG,9,CTATCACAT,True
2,VH_003,CCCGGG,6,CCCGGG,False
3,VH_004,ATGTGA,6,TCACAT,True


In [5]:
# Cell 4: Feature Engineering for Machine Learning

def extract_kmers(sequence, k=3):
    """Counts all k-mers of length k in a given sequence."""
    kmer_counts = {}
    for i in range(len(sequence) - k + 1):
        kmer = sequence[i:i+k]
        if kmer in kmer_counts:
            kmer_counts[kmer] += 1
        else:
            kmer_counts[kmer] = 1
    return kmer_counts

# 1. Apply the function (returns a column of dictionaries)
kmer_dicts = df['sequence'].apply(lambda seq: extract_kmers(seq, k=3))

# 2. Expand those dictionaries into their own distinct columns 
# (Pandas will automatically align them and fill missing k-mers with 0)
kmer_features = kmer_dicts.apply(pd.Series).fillna(0).astype(int)

# 3. Join these new numerical features to our original dataframe
ml_ready_df = pd.concat([df, kmer_features], axis=1)

# View the final ML-ready dataset
ml_ready_df

,clone_id,sequence,length,rev_comp,has_stop_codon,ATG,TGC,GCG,CGT,GTA,...,TGT,GTG,TGA,GAT,ATA,TAG,CCC,CCG,CGG,GGG
0,VH_001,ATGCGTAN,8,NTACGCAT,False,1,1,1,1,1,...,0,0,0,0,0,0,0,0,0,0
1,VH_002,ATGTGATAG,9,CTATCACAT,True,1,0,0,0,0,...,1,1,1,1,1,1,0,0,0,0
2,VH_003,CCCGGG,6,CCCGGG,False,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,1
3,VH_004,ATGTGA,6,TCACAT,True,1,0,0,0,0,...,1,1,1,0,0,0,0,0,0,0


In [6]:
# Cell 5: Train Your First Machine Learning Model
from sklearn.ensemble import RandomForestClassifier

# 1. Define your inputs (X) and your output (y)
# X is the numerical matrix (our k-mer counts)
# y is the label we want the AI to predict (True/False for stop codons)
X = kmer_features
y = ml_ready_df['has_stop_codon']

# 2. Initialize the model
# random_state ensures we get the same results every time we run it
model = RandomForestClassifier(random_state=42)

# 3. Train the model (this is where the "learning" happens)
model.fit(X, y)

# 4. Let's test it by asking it to predict on our exact training data
predictions = model.predict(X)

# Add the AI's predictions back into our dataframe to compare
ml_ready_df['AI_Prediction'] = predictions

# View the final result
ml_ready_df[['sequence', 'has_stop_codon', 'AI_Prediction']]

,sequence,has_stop_codon,AI_Prediction
0,ATGCGTAN,False,False
1,ATGTGATAG,True,True
2,CCCGGG,False,False
3,ATGTGA,True,True
